In [ ]:
!pip install -q transformers datasets sentencepiece scikit-learn accelerate

In [ ]:

# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup
)

from torch.utils.data import DataLoader
from torch.optim import AdamW



In [ ]:
# ============================================================
# IMPORT PYTORCH
# ============================================================

import torch

print("PyTorch imported successfully.")
print("PyTorch version:", torch.__version__)

PyTorch imported successfully.
PyTorch version: 2.11.0+cu128


In [ ]:
# ============================================================
# 3. CHECK GPU
# ============================================================

print("=" * 70)
print("SYSTEM INFORMATION")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

    device = torch.device("cuda")

else:

    print("WARNING: CUDA is not available.")
    print("Training will use CPU.")

    device = torch.device("cpu")

print("Using device:", device)

SYSTEM INFORMATION
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Using device: cuda


In [3]:
from huggingface_hub import login
login()

In [ ]:

from transformers import AutoModel
model = AutoModel.from_pretrained("ai4bharat/indic-bert", device_map="auto")

config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  135MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
predictions.decoder.bias         | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# ============================================================
# INDICBERT - DATASET A
# Dataset A -> Dataset A | 80:20 Split
# ============================================================

!pip install -q transformers datasets sentencepiece scikit-learn accelerate

import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

from torch.utils.data import DataLoader
from torch.optim import AdamW


# ============================================================
# 1. SETTINGS
# ============================================================

SEED = 42
MODEL_NAME = "ai4bharat/indic-bert"

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 2. LOAD DATASET A
# ============================================================

df = pd.read_csv("/content/Dataset_A_clean.csv")

print("\nDataset shape:", df.shape)
print("\nClass distribution:")
print(df["Is Fake"].value_counts().sort_index())

# DO NOT REMOVE DUPLICATES
df["Tamil"] = df["Tamil"].fillna("").astype(str)
df["Is Fake"] = df["Is Fake"].astype(int)

df = df.rename(
    columns={
        "Tamil": "text",
        "Is Fake": "labels"
    }
)


# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["labels"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nTraining samples:", len(train_df))
print("Testing samples:", len(test_df))

print("\nTraining distribution:")
print(train_df["labels"].value_counts().sort_index())

print("\nTesting distribution:")
print(test_df["labels"].value_counts().sort_index())


# ============================================================
# 4. CLASS WEIGHTS
# ============================================================

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["labels"]
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float
).to(device)

print("\nClass weights:")
print("Class 0:", weights[0])
print("Class 1:", weights[1])


# ============================================================
# 5. HUGGING FACE DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df[["text", "labels"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "labels"]],
    preserve_index=False
)


# ============================================================
# 6. INDICBERT TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    keep_accents=True
)


def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )


train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])


# ============================================================
# 7. DATALOADERS
# ============================================================

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=False,
    return_tensors="pt"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0
)


# ============================================================
# 8. LOAD INDICBERT
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model = model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# 9. TRAINING
# ============================================================

print("\n" + "=" * 60)
print("INDICBERT TRAINING")
print("=" * 60)

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(
            outputs.logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_accuracy = correct / total

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Accuracy: {train_accuracy:.4f}"
    )


# ============================================================
# 10. TESTING
# ============================================================

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        y_true.extend(
            labels.cpu().numpy()
        )

        y_pred.extend(
            predictions.cpu().numpy()
        )


# ============================================================
# 11. RESULTS
# ============================================================

accuracy = accuracy_score(
    y_true,
    y_pred
)

print("\n" + "=" * 60)
print("INDICBERT DATASET A RESULTS")
print("=" * 60)

print(f"\nTest Accuracy: {accuracy:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Real", "Fake"],
        digits=2,
        zero_division=0
    )
)

print("Confusion Matrix:")

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

print(cm)

print("\nMatrix:")
print("             Predicted")
print("             Real   Fake")
print(f"Actual Real  {cm[0][0]:5d}  {cm[0][1]:5d}")
print(f"Actual Fake  {cm[1][0]:5d}  {cm[1][1]:5d}")


# ============================================================
# 12. SAVE MODEL
# ============================================================

model.save_pretrained(
    "/content/indicbert_dataset_A"
)

tokenizer.save_pretrained(
    "/content/indicbert_dataset_A"
)

print("\nModel saved to:")
print("/content/indicbert_dataset_A")

Using device: cuda
GPU: Tesla T4

Dataset shape: (14564, 2)

Class distribution:
Is Fake
0    11662
1     2902
Name: count, dtype: int64

Training samples: 11651
Testing samples: 2913

Training distribution:
labels
0    9329
1    2322
Name: count, dtype: int64

Testing distribution:
labels
0    2333
1     580
Name: count, dtype: int64

Class weights:
Class 0: 0.6244506377961196
Class 1: 2.5088285960378984


Map:   0%|          | 0/11651 [00:00<?, ? examples/s]

Map:   0%|          | 0/2913 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     | 
---------------------------------+------------+-
predictions.decoder.bias         | UNEXPECTED | 
predictions.decoder.weight       | UNEXPECTED | 
sop_classifier.classifier.bias   | UNEXPECTED | 
predictions.LayerNorm.bias       | UNEXPECTED | 
predictions.bias                 | UNEXPECTED | 
predictions.LayerNorm.weight     | UNEXPECTED | 
sop_classifier.classifier.weight | UNEXPECTED | 
predictions.dense.weight         | UNEXPECTED | 
predictions.dense.bias           | UNEXPECTED | 
classifier.weight                | MISSING    | 
classifier.bias                  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



INDICBERT TRAINING
Epoch 1/3 | Loss: 0.2051 | Accuracy: 0.9710
Epoch 2/3 | Loss: 0.1449 | Accuracy: 0.9821
Epoch 3/3 | Loss: 0.1114 | Accuracy: 0.9847

INDICBERT DATASET A RESULTS

Test Accuracy: 0.9870
Test Accuracy: 98.70%

Classification Report:
              precision    recall  f1-score   support

        Real       0.99      1.00      0.99      2333
        Fake       0.99      0.94      0.97       580

    accuracy                           0.99      2913
   macro avg       0.99      0.97      0.98      2913
weighted avg       0.99      0.99      0.99      2913

Confusion Matrix:
[[2327    6]
 [  32  548]]

Matrix:
             Predicted
             Real   Fake
Actual Real   2327      6
Actual Fake     32    548


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to:
/content/indicbert_dataset_A


In [ ]:
# ============================================================
# INDICBERT - EXPERIMENT 2
# DATASET B -> DATASET B
# 80:20 STRATIFIED SPLIT
# ============================================================

!pip install -q transformers datasets sentencepiece scikit-learn accelerate

import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

from torch.utils.data import DataLoader
from torch.optim import AdamW


# ============================================================
# 1. SETTINGS
# ============================================================

SEED = 42

MODEL_NAME = "ai4bharat/indic-bert"

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 2. LOAD DATASET B
# ============================================================

df = pd.read_csv("/content/Dataset_B_clean.csv")

print("\nDataset shape:", df.shape)

print("\nClass distribution:")
print(df["Authenticity"].value_counts().sort_index())

# DO NOT REMOVE DUPLICATES
df["News"] = df["News"].fillna("").astype(str)
df["Authenticity"] = df["Authenticity"].astype(int)

df = df.rename(
    columns={
        "News": "text",
        "Authenticity": "labels"
    }
)


# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["labels"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nTraining samples:", len(train_df))
print("Testing samples:", len(test_df))

print("\nTraining distribution:")
print(train_df["labels"].value_counts().sort_index())

print("\nTesting distribution:")
print(test_df["labels"].value_counts().sort_index())


# ============================================================
# 4. CLASS WEIGHTS
# ============================================================

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["labels"]
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float
).to(device)

print("\nClass weights:")
print(f"Class 0: {weights[0]:.6f}")
print(f"Class 1: {weights[1]:.6f}")


# ============================================================
# 5. HUGGING FACE DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df[["text", "labels"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "labels"]],
    preserve_index=False
)


# ============================================================
# 6. INDICBERT TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    keep_accents=True
)


def tokenize(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )


train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])


# ============================================================
# 7. DATALOADERS
# ============================================================

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0
)


# ============================================================
# 8. LOAD INDICBERT
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model = model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# 9. TRAINING
# ============================================================

print("\n" + "=" * 60)
print("INDICBERT - DATASET B TRAINING")
print("=" * 60)

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(
            outputs.logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_accuracy = correct / total

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Accuracy: {train_accuracy * 100:.2f}%"
    )


# ============================================================
# 10. TESTING
# ============================================================

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        y_true.extend(
            labels.cpu().numpy()
        )

        y_pred.extend(
            predictions.cpu().numpy()
        )


# ============================================================
# 11. FINAL RESULTS
# ============================================================

accuracy = accuracy_score(
    y_true,
    y_pred
)

print("\n" + "=" * 60)
print("INDICBERT DATASET B RESULTS")
print("=" * 60)

print(
    f"\nTest Accuracy: {accuracy * 100:.2f}%"
)


# ============================================================
# 12. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Real", "Fake"],
        digits=2,
        zero_division=0
    )
)


# ============================================================
# 13. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

print("Confusion Matrix:")
print(cm)

print("\nMatrix:")
print("             Predicted")
print("             Real   Fake")
print(
    f"Actual Real  {cm[0][0]:5d}  {cm[0][1]:5d}"
)
print(
    f"Actual Fake  {cm[1][0]:5d}  {cm[1][1]:5d}"
)


# ============================================================
# 14. SAVE MODEL
# ============================================================

model.save_pretrained(
    "/content/indicbert_dataset_B"
)

tokenizer.save_pretrained(
    "/content/indicbert_dataset_B"
)

print("\nModel saved to:")
print("/content/indicbert_dataset_B")

Using device: cuda
GPU: Tesla T4

Dataset shape: (5226, 2)

Class distribution:
Authenticity
0    2324
1    2902
Name: count, dtype: int64

Training samples: 4180
Testing samples: 1046

Training distribution:
labels
0    1859
1    2321
Name: count, dtype: int64

Testing distribution:
labels
0    465
1    581
Name: count, dtype: int64

Class weights:
Class 0: 1.124260
Class 1: 0.900474


Map:   0%|          | 0/4180 [00:00<?, ? examples/s]

Map:   0%|          | 0/1046 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     | 
---------------------------------+------------+-
predictions.LayerNorm.bias       | UNEXPECTED | 
predictions.LayerNorm.weight     | UNEXPECTED | 
predictions.dense.bias           | UNEXPECTED | 
sop_classifier.classifier.weight | UNEXPECTED | 
predictions.decoder.weight       | UNEXPECTED | 
predictions.bias                 | UNEXPECTED | 
predictions.decoder.bias         | UNEXPECTED | 
sop_classifier.classifier.bias   | UNEXPECTED | 
predictions.dense.weight         | UNEXPECTED | 
classifier.bias                  | MISSING    | 
classifier.weight                | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



INDICBERT - DATASET B TRAINING
Epoch 1/3 | Loss: 0.2894 | Accuracy: 91.17%
Epoch 2/3 | Loss: 0.1565 | Accuracy: 95.57%
Epoch 3/3 | Loss: 0.1113 | Accuracy: 96.82%

INDICBERT DATASET B RESULTS

Test Accuracy: 97.71%

Classification Report:
              precision    recall  f1-score   support

        Real       0.96      0.99      0.97       465
        Fake       0.99      0.97      0.98       581

    accuracy                           0.98      1046
   macro avg       0.98      0.98      0.98      1046
weighted avg       0.98      0.98      0.98      1046

Confusion Matrix:
[[459   6]
 [ 18 563]]

Matrix:
             Predicted
             Real   Fake
Actual Real    459      6
Actual Fake     18    563


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to:
/content/indicbert_dataset_B


In [ ]:
# ============================================================
# INDICBERT EXPERIMENT 3: DATASET A → DATASET B
# ============================================================

!pip install -q transformers datasets sentencepiece scikit-learn accelerate

import torch
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. LOAD DATA
# ============================================================

df_A = pd.read_csv("/content/Dataset_A_clean.csv")
df_B = pd.read_csv("/content/Dataset_B_clean.csv")

print("\nDataset A shape:", df_A.shape)
print("Dataset B shape:", df_B.shape)

# ============================================================
# 3. PREPARE DATA
# ============================================================

df_A = df_A.rename(columns={
    "Tamil": "text",
    "Is Fake": "labels"
})

df_B = df_B.rename(columns={
    "News": "text",
    "Authenticity": "labels"
})

# NO duplicate removal

df_A = df_A[["text", "labels"]].copy()
df_B = df_B[["text", "labels"]].copy()

df_A["labels"] = df_A["labels"].astype(int)
df_B["labels"] = df_B["labels"].astype(int)

print("\nDataset A class distribution:")
print(df_A["labels"].value_counts().sort_index())

print("\nDataset B class distribution:")
print(df_B["labels"].value_counts().sort_index())

# ============================================================
# 4. A → B
# ============================================================

train_df = df_A
test_df = df_B

print("\nTraining samples:", len(train_df))
print("External test samples:", len(test_df))

print("\nTraining distribution:")
print(train_df["labels"].value_counts().sort_index())

print("\nExternal test distribution:")
print(test_df["labels"].value_counts().sort_index())

# ============================================================
# 5. CLASS WEIGHTS
# ============================================================

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["labels"]
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float
).to(device)

print("\nClass weights:")
print(f"Class 0: {weights[0]:.6f}")
print(f"Class 1: {weights[1]:.6f}")

# ============================================================
# 6. HUGGING FACE DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

# ============================================================
# 7. TOKENIZER
# ============================================================

MODEL_NAME = "ai4bharat/indic-bert"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    keep_accents=True
)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
       padding="max_length",
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

# IMPORTANT:
# Do NOT use set_format(type="torch")
# This avoids the torchvision VideoReader problem.

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

# ============================================================
# 8. DATALOADERS
# ============================================================

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0
)

print("\nDataLoaders created successfully.")

# ============================================================
# 9. MODEL
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

print("IndicBERT model loaded successfully.")

# ============================================================
# 10. OPTIMIZER + LOSS
# ============================================================

EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

criterion = CrossEntropyLoss(
    weight=class_weights
)

# ============================================================
# 11. TRAINING
# ============================================================

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(
            outputs.logits,
            labels
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    accuracy = correct / total * 100
    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {avg_loss:.4f} | "
        f"Accuracy: {accuracy:.2f}%"
    )

# ============================================================
# 12. TEST ON ALL DATASET B
# ============================================================

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

# ============================================================
# 13. RESULTS
# ============================================================

accuracy = (
    np.array(all_predictions) ==
    np.array(all_labels)
).mean() * 100

print("\n" + "=" * 55)
print("INDICBERT DATASET A → DATASET B RESULTS")
print("=" * 55)

print(f"\nTest Accuracy: {accuracy:.2f}%")

print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Real", "Fake"],
        digits=2
    )
)

print("Confusion Matrix:")

print(
    confusion_matrix(
        all_labels,
        all_predictions
    )
)

# ============================================================
# 14. SAVE MODEL
# ============================================================

SAVE_PATH = "/content/indicbert_A_to_B"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("\nModel saved to:")
print(SAVE_PATH)

Device: cuda
GPU: Tesla T4

Dataset A shape: (14564, 2)
Dataset B shape: (5226, 2)

Dataset A class distribution:
labels
0    11662
1     2902
Name: count, dtype: int64

Dataset B class distribution:
labels
0    2324
1    2902
Name: count, dtype: int64

Training samples: 14564
External test samples: 5226

Training distribution:
labels
0    11662
1     2902
Name: count, dtype: int64

External test distribution:
labels
0    2324
1    2902
Name: count, dtype: int64

Class weights:
Class 0: 0.624421
Class 1: 2.509304


Map:   0%|          | 0/14564 [00:00<?, ? examples/s]

Map:   0%|          | 0/5226 [00:00<?, ? examples/s]


DataLoaders created successfully.


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     | 
---------------------------------+------------+-
predictions.decoder.bias         | UNEXPECTED | 
predictions.decoder.weight       | UNEXPECTED | 
sop_classifier.classifier.bias   | UNEXPECTED | 
predictions.LayerNorm.bias       | UNEXPECTED | 
predictions.bias                 | UNEXPECTED | 
predictions.LayerNorm.weight     | UNEXPECTED | 
sop_classifier.classifier.weight | UNEXPECTED | 
predictions.dense.weight         | UNEXPECTED | 
predictions.dense.bias           | UNEXPECTED | 
classifier.weight                | MISSING    | 
classifier.bias                  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


IndicBERT model loaded successfully.
Epoch 1/3 | Loss: 0.1654 | Accuracy: 96.17%
Epoch 2/3 | Loss: 0.1061 | Accuracy: 97.66%
Epoch 3/3 | Loss: 0.0889 | Accuracy: 98.01%

INDICBERT DATASET A → DATASET B RESULTS

Test Accuracy: 96.57%

Classification Report:
              precision    recall  f1-score   support

        Real       0.96      0.97      0.96      2324
        Fake       0.97      0.96      0.97      2902

    accuracy                           0.97      5226
   macro avg       0.96      0.97      0.97      5226
weighted avg       0.97      0.97      0.97      5226

Confusion Matrix:
[[2249   75]
 [ 104 2798]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to:
/content/indicbert_A_to_B


In [ ]:
# ============================================================
# INDICBERT — DATASET B → DATASET A
# Fixed Padding Version
# ============================================================

# ============================================================
# 1. INSTALL LIBRARIES
# ============================================================

!pip install -q transformers datasets accelerate sentencepiece scikit-learn

# ============================================================
# 2. IMPORTS
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

# ============================================================
# 3. SEED
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ============================================================
# 4. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 5. LOAD DATASETS
# ============================================================

dataset_A_path = "/content/Dataset_A_clean.csv"
dataset_B_path = "/content/Dataset_B_clean.csv"

df_A = pd.read_csv(dataset_A_path)
df_B = pd.read_csv(dataset_B_path)

print("\nDataset A shape:", df_A.shape)
print("Dataset B shape:", df_B.shape)

# ============================================================
# 6. PREPARE DATASET A
# ============================================================

df_A = df_A[["Tamil", "Is Fake"]].copy()

df_A = df_A.rename(columns={
    "Tamil": "text",
    "Is Fake": "label"
})

# ============================================================
# 7. PREPARE DATASET B
# ============================================================

df_B = df_B[["News", "Authenticity"]].copy()

df_B = df_B.rename(columns={
    "News": "text",
    "Authenticity": "label"
})

# ============================================================
# IMPORTANT:
# NO DUPLICATE REMOVAL
# ============================================================

print("\nDataset A class distribution:")
print(df_A["label"].value_counts().sort_index())

print("\nDataset B class distribution:")
print(df_B["label"].value_counts().sort_index())

# ============================================================
# 8. B → A EXPERIMENT
# ============================================================
#
# TRAIN = ALL DATASET B
# TEST  = ALL DATASET A
#
# No 80:20 split.
# ============================================================

X_train = df_B["text"].astype(str).tolist()
y_train = df_B["label"].astype(int).tolist()

X_test = df_A["text"].astype(str).tolist()
y_test = df_A["label"].astype(int).tolist()

print("\n====================================")
print("B → A GENERALISATION")
print("====================================")

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining class distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTesting class distribution:")
print(pd.Series(y_test).value_counts().sort_index())

# ============================================================
# 9. CLASS WEIGHTS
# ============================================================

classes = np.array([0, 1])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

print("\nClass weights:")

for cls, weight in zip(classes, class_weights):
    print(f"Class {cls}: {weight.item():.6f}")

# ============================================================
# 10. INDICBERT TOKENIZER
# ============================================================

MODEL_NAME = "ai4bharat/indic-bert"

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    keep_accents=True
)

# ============================================================
# 11. CREATE HUGGING FACE DATASETS
# ============================================================

train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

# ============================================================
# 12. TOKENIZATION
# ============================================================
#
# IMPORTANT:
# Fixed padding to MAX_LENGTH = 128
#
# This matches the padding approach used in your mBERT code.
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

print("\nTokenizing training data...")

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenizing testing data...")

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

# ============================================================
# 13. REMOVE ORIGINAL TEXT COLUMN
# ============================================================

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

# ============================================================
# 14. DATA COLLATOR
# ============================================================
#
# Since everything is already padded to 128,
# the collator simply converts the batch to PyTorch tensors.
#
# No Dataset.set_format(type="torch") is used.
# This avoids the torchvision VideoReader issue.
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=False,
    return_tensors="pt"
)

# ============================================================
# 15. DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

print("\nTrain batches:", len(train_loader))
print("Test batches :", len(test_loader))

# ============================================================
# 16. LOAD INDICBERT MODEL
# ============================================================

print("\nLoading IndicBERT model...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

# ============================================================
# 17. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

# ============================================================
# 18. LOSS FUNCTION
# ============================================================

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights
)

# ============================================================
# 19. TRAINING
# ============================================================

print("\n====================================")
print("TRAINING")
print("====================================")

model.train()

for epoch in range(EPOCHS):

    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    avg_loss = total_loss / len(train_loader)
    train_accuracy = correct / total

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"- Loss: {avg_loss:.4f} "
        f"- Accuracy: {train_accuracy:.4f}"
    )

# ============================================================
# 20. EVALUATION ON DATASET A
# ============================================================

print("\n====================================")
print("EVALUATION — DATASET A")
print("====================================")

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

# ============================================================
# 21. ACCURACY
# ============================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print(f"\nTest Accuracy: {accuracy:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# ============================================================
# 22. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Real (0)",
            "Fake (1)"
        ],
        digits=2
    )
)

# ============================================================
# 23. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nConfusion Matrix:")
print(cm)

# ============================================================
# 24. SAVE MODEL
# ============================================================

SAVE_PATH = "/content/indicbert_B_to_A"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("\nModel saved to:")
print(SAVE_PATH)

# ============================================================
# 25. FINAL SUMMARY
# ============================================================

print("\n====================================")
print("FINAL RESULT — INDICBERT B → A")
print("====================================")



Using device: cuda
GPU: Tesla T4

Dataset A shape: (14564, 2)
Dataset B shape: (5226, 2)

Dataset A class distribution:
label
0    11662
1     2902
Name: count, dtype: int64

Dataset B class distribution:
label
0    2324
1    2902
Name: count, dtype: int64

B → A GENERALISATION
Training samples: 5226
Testing samples : 14564

Training class distribution:
0    2324
1    2902
Name: count, dtype: int64

Testing class distribution:
0    11662
1     2902
Name: count, dtype: int64

Class weights:
Class 0: 1.124355
Class 1: 0.900414

Loading tokenizer...

Tokenizing training data...


Map:   0%|          | 0/5226 [00:00<?, ? examples/s]

Tokenizing testing data...


Map:   0%|          | 0/14564 [00:00<?, ? examples/s]


Train batches: 327
Test batches : 911

Loading IndicBERT model...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     | 
---------------------------------+------------+-
predictions.LayerNorm.bias       | UNEXPECTED | 
predictions.LayerNorm.weight     | UNEXPECTED | 
predictions.dense.bias           | UNEXPECTED | 
sop_classifier.classifier.weight | UNEXPECTED | 
predictions.decoder.weight       | UNEXPECTED | 
predictions.bias                 | UNEXPECTED | 
predictions.decoder.bias         | UNEXPECTED | 
sop_classifier.classifier.bias   | UNEXPECTED | 
predictions.dense.weight         | UNEXPECTED | 
classifier.bias                  | MISSING    | 
classifier.weight                | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



TRAINING
Epoch 1/3 - Loss: 0.3229 - Accuracy: 0.8907
Epoch 2/3 - Loss: 0.1307 - Accuracy: 0.9518
Epoch 3/3 - Loss: 0.0992 - Accuracy: 0.9650

EVALUATION — DATASET A

Test Accuracy: 0.9001
Test Accuracy: 90.01%

Classification Report:
              precision    recall  f1-score   support

    Real (0)       0.98      0.89      0.93     11662
    Fake (1)       0.68      0.94      0.79      2902

    accuracy                           0.90     14564
   macro avg       0.83      0.91      0.86     14564
weighted avg       0.92      0.90      0.91     14564


Confusion Matrix:
[[10382  1280]
 [  175  2727]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to:
/content/indicbert_B_to_A

FINAL RESULT — INDICBERT B → A


In [ ]:
# ============================================================
# INDICBERT EXPERIMENT 5:
# COMBINED DATASET - IMBALANCED
# ============================================================

!pip install -q transformers datasets sentencepiece scikit-learn accelerate

import torch
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix


# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 2. RANDOM SEED
# ============================================================

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 3. LOAD DATA
# ============================================================

df_A = pd.read_csv("/content/Dataset_A_clean.csv")
df_B = pd.read_csv("/content/Dataset_B_clean.csv")

print("\nDataset A shape:", df_A.shape)
print("Dataset B shape:", df_B.shape)


# ============================================================
# 4. PREPARE DATASET A
# ============================================================

df_A = df_A.rename(columns={
    "Tamil": "text",
    "Is Fake": "labels"
})

df_A = df_A[["text", "labels"]].copy()

df_A["labels"] = df_A["labels"].astype(int)


# ============================================================
# 5. PREPARE DATASET B
# ============================================================

df_B = df_B.rename(columns={
    "News": "text",
    "Authenticity": "labels"
})

df_B = df_B[["text", "labels"]].copy()

df_B["labels"] = df_B["labels"].astype(int)


# ============================================================
# IMPORTANT:
# NO DUPLICATE REMOVAL
# ============================================================

print("\nNo duplicate removal performed.")


# ============================================================
# 6. DISPLAY INDIVIDUAL CLASS DISTRIBUTIONS
# ============================================================

print("\nDataset A class distribution:")
print(
    df_A["labels"]
    .value_counts()
    .sort_index()
)

print("\nDataset B class distribution:")
print(
    df_B["labels"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 7. COMBINE DATASET A + DATASET B
# ============================================================

combined_df = pd.concat(
    [df_A, df_B],
    ignore_index=True
)

print("\n" + "=" * 60)
print("COMBINED DATASET")
print("=" * 60)

print("\nCombined shape:", combined_df.shape)

print("\nCombined class distribution:")
print(
    combined_df["labels"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 8. 80:20 STRATIFIED TRAIN/TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    combined_df,
    test_size=0.20,
    random_state=SEED,
    stratify=combined_df["labels"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


# ============================================================
# 9. DISPLAY SPLIT INFORMATION
# ============================================================

print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print("\nTraining samples:", len(train_df))
print("Testing samples:", len(test_df))

print("\nTraining distribution:")
print(
    train_df["labels"]
    .value_counts()
    .sort_index()
)

print("\nTesting distribution:")
print(
    test_df["labels"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 10. CLASS WEIGHTS
# ============================================================

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["labels"]
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float
).to(device)

print("\nClass weights:")
print(f"Class 0: {weights[0]:.6f}")
print(f"Class 1: {weights[1]:.6f}")


# ============================================================
# 11. CONVERT TO HUGGING FACE DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)


# ============================================================
# 12. TOKENIZER
# ============================================================

MODEL_NAME = "ai4bharat/indic-bert"

MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    keep_accents=True
)


# ============================================================
# 13. TOKENIZATION
# ============================================================

def tokenize_function(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )


train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)


# ============================================================
# 14. REMOVE TEXT COLUMN
# ============================================================

train_dataset = train_dataset.remove_columns(
    ["text"]
)

test_dataset = test_dataset.remove_columns(
    ["text"]
)


# ============================================================
# 15. DATA COLLATOR
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=False,
    return_tensors="pt"
)


# ============================================================
# 16. DATALOADERS
# ============================================================

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0
)

print("\nDataLoaders created successfully.")

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))


# ============================================================
# 17. LOAD INDICBERT
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

print("\nIndicBERT model loaded successfully.")


# ============================================================
# 18. OPTIMIZER + LOSS
# ============================================================

EPOCHS = 3

LEARNING_RATE = 2e-5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

criterion = CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# 19. TRAINING
# ============================================================

print("\n" + "=" * 60)
print("INDICBERT TRAINING - COMBINED IMBALANCED DATASET")
print("=" * 60)

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(
            outputs.logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    accuracy = (
        correct / total
    ) * 100

    avg_loss = (
        total_loss / len(train_loader)
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {avg_loss:.4f} | "
        f"Accuracy: {accuracy:.2f}%"
    )


# ============================================================
# 20. TESTING
# ============================================================

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )


# ============================================================
# 21. CALCULATE ACCURACY
# ============================================================

accuracy = (
    np.array(all_predictions)
    ==
    np.array(all_labels)
).mean() * 100


# ============================================================
# 22. FINAL RESULTS
# ============================================================

print("\n" + "=" * 60)
print("INDICBERT COMBINED IMBALANCED DATASET RESULTS")
print("=" * 60)

print(
    f"\nTest Accuracy: {accuracy:.2f}%"
)

print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Real",
            "Fake"
        ],
        digits=2
    )
)


# ============================================================
# 23. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("Confusion Matrix:")

print(cm)

print("\nMatrix:")
print(
    "             Predicted"
)

print(
    "             Real   Fake"
)

print(
    f"Actual Real   {cm[0][0]:4d}   {cm[0][1]:4d}"
)

print(
    f"Actual Fake   {cm[1][0]:4d}   {cm[1][1]:4d}"
)


# ============================================================
# 24. SAVE MODEL
# ============================================================

SAVE_PATH = "/content/indicbert_combined_imbalanced"

model.save_pretrained(
    SAVE_PATH
)

tokenizer.save_pretrained(
    SAVE_PATH
)

print("\nModel saved to:")
print(SAVE_PATH)


Using device: cuda
GPU: Tesla T4

Dataset A shape: (14564, 2)
Dataset B shape: (5226, 2)

No duplicate removal performed.

Dataset A class distribution:
labels
0    11662
1     2902
Name: count, dtype: int64

Dataset B class distribution:
labels
0    2324
1    2902
Name: count, dtype: int64

COMBINED DATASET

Combined shape: (19790, 2)

Combined class distribution:
labels
0    13986
1     5804
Name: count, dtype: int64

TRAIN / TEST SPLIT

Training samples: 15832
Testing samples: 3958

Training distribution:
labels
0    11189
1     4643
Name: count, dtype: int64

Testing distribution:
labels
0    2797
1    1161
Name: count, dtype: int64

Class weights:
Class 0: 0.707481
Class 1: 1.704932


Map:   0%|          | 0/15832 [00:00<?, ? examples/s]

Map:   0%|          | 0/3958 [00:00<?, ? examples/s]


DataLoaders created successfully.
Training batches: 990
Testing batches: 248


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     | 
---------------------------------+------------+-
predictions.decoder.bias         | UNEXPECTED | 
predictions.decoder.weight       | UNEXPECTED | 
sop_classifier.classifier.bias   | UNEXPECTED | 
predictions.LayerNorm.bias       | UNEXPECTED | 
predictions.bias                 | UNEXPECTED | 
predictions.LayerNorm.weight     | UNEXPECTED | 
sop_classifier.classifier.weight | UNEXPECTED | 
predictions.dense.weight         | UNEXPECTED | 
predictions.dense.bias           | UNEXPECTED | 
classifier.weight                | MISSING    | 
classifier.bias                  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



IndicBERT model loaded successfully.

INDICBERT TRAINING - COMBINED IMBALANCED DATASET
Epoch 1/3 | Loss: 0.1662 | Accuracy: 95.19%
Epoch 2/3 | Loss: 0.0942 | Accuracy: 97.43%
Epoch 3/3 | Loss: 0.0681 | Accuracy: 97.91%

INDICBERT COMBINED IMBALANCED DATASET RESULTS

Test Accuracy: 98.21%

Classification Report:
              precision    recall  f1-score   support

        Real       0.99      0.99      0.99      2797
        Fake       0.97      0.97      0.97      1161

    accuracy                           0.98      3958
   macro avg       0.98      0.98      0.98      3958
weighted avg       0.98      0.98      0.98      3958

Confusion Matrix:
[[2758   39]
 [  32 1129]]

Matrix:
             Predicted
             Real   Fake
Actual Real   2758     39
Actual Fake     32   1129


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to:
/content/indicbert_combined_imbalanced


In [4]:
# ============================================================
# INDICBERT EXPERIMENT 6:
# COMBINED DATASET - BALANCED
# ============================================================

!pip install -q transformers datasets sentencepiece scikit-learn accelerate

import torch
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 2. RANDOM SEED
# ============================================================

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 3. LOAD DATA
# ============================================================

df_A = pd.read_csv(
    "/content/Dataset_A_clean.csv"
)

df_B = pd.read_csv(
    "/content/Dataset_B_clean.csv"
)

print("\nDataset A shape:", df_A.shape)
print("Dataset B shape:", df_B.shape)


# ============================================================
# 4. PREPARE DATASET A
# ============================================================

df_A = df_A.rename(columns={
    "Tamil": "text",
    "Is Fake": "labels"
})

df_A = df_A[
    ["text", "labels"]
].copy()

df_A["labels"] = df_A["labels"].astype(int)


# ============================================================
# 5. PREPARE DATASET B
# ============================================================

df_B = df_B.rename(columns={
    "News": "text",
    "Authenticity": "labels"
})

df_B = df_B[
    ["text", "labels"]
].copy()

df_B["labels"] = df_B["labels"].astype(int)


# ============================================================
# IMPORTANT:
# NO DUPLICATE REMOVAL
# ============================================================

print("\nNo duplicate removal performed.")


# ============================================================
# 6. COMBINE DATASET A + DATASET B
# ============================================================

combined_df = pd.concat(
    [df_A, df_B],
    ignore_index=True
)

print("\n" + "=" * 60)
print("COMBINED DATASET BEFORE BALANCING")
print("=" * 60)

print("\nCombined shape:", combined_df.shape)

print("\nClass distribution:")
print(
    combined_df["labels"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 7. BALANCE THE COMBINED DATASET
# ============================================================

class_0 = combined_df[
    combined_df["labels"] == 0
]

class_1 = combined_df[
    combined_df["labels"] == 1
]

print("\nClass 0 samples:", len(class_0))
print("Class 1 samples:", len(class_1))


# Use the smaller class size
min_class_size = min(
    len(class_0),
    len(class_1)
)

print(
    "\nBalancing both classes to:",
    min_class_size
)


# ============================================================
# UNDERSAMPLE CLASS 0
# KEEP ALL CLASS 1
# ============================================================

class_0_balanced = class_0.sample(
    n=min_class_size,
    random_state=SEED
)

class_1_balanced = class_1.sample(
    n=min_class_size,
    random_state=SEED
)


# ============================================================
# CREATE BALANCED DATASET
# ============================================================

balanced_df = pd.concat(
    [
        class_0_balanced,
        class_1_balanced
    ],
    ignore_index=True
)


# Shuffle balanced dataset
balanced_df = balanced_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)


# ============================================================
# 8. DISPLAY BALANCED DATASET
# ============================================================

print("\n" + "=" * 60)
print("BALANCED COMBINED DATASET")
print("=" * 60)

print("\nBalanced shape:", balanced_df.shape)

print("\nBalanced class distribution:")
print(
    balanced_df["labels"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 9. 80:20 STRATIFIED TRAIN/TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    balanced_df,
    test_size=0.20,
    random_state=SEED,
    stratify=balanced_df["labels"]
)

train_df = train_df.reset_index(
    drop=True
)

test_df = test_df.reset_index(
    drop=True
)


# ============================================================
# 10. DISPLAY TRAIN / TEST INFORMATION
# ============================================================

print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print("\nTraining samples:", len(train_df))
print("Testing samples:", len(test_df))

print("\nTraining distribution:")
print(
    train_df["labels"]
    .value_counts()
    .sort_index()
)

print("\nTesting distribution:")
print(
    test_df["labels"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 11. HUGGING FACE DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)


# ============================================================
# 12. TOKENIZER
# ============================================================

MODEL_NAME = "ai4bharat/indic-bert"

MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    keep_accents=True
)


# ============================================================
# 13. TOKENIZATION
# ============================================================

def tokenize_function(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )


train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)


# ============================================================
# 14. REMOVE TEXT COLUMN
# ============================================================

train_dataset = train_dataset.remove_columns(
    ["text"]
)

test_dataset = test_dataset.remove_columns(
    ["text"]
)


# ============================================================
# 15. DATA COLLATOR
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=False,
    return_tensors="pt"
)


# ============================================================
# 16. DATALOADERS
# ============================================================

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0
)

print("\nDataLoaders created successfully.")

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Testing batches:",
    len(test_loader)
)


# ============================================================
# 17. LOAD INDICBERT
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

print(
    "\nIndicBERT model loaded successfully."
)


# ============================================================
# 18. OPTIMIZER + LOSS
# ============================================================

EPOCHS = 3

LEARNING_RATE = 2e-5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

# Dataset is balanced,
# so no class weights are required.

criterion = CrossEntropyLoss()


# ============================================================
# 19. TRAINING
# ============================================================

print("\n" + "=" * 60)
print("INDICBERT TRAINING - COMBINED BALANCED DATASET")
print("=" * 60)

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:

        input_ids = batch[
            "input_ids"
        ].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        labels = batch[
            "labels"
        ].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(
            outputs.logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    accuracy = (
        correct / total
    ) * 100

    avg_loss = (
        total_loss /
        len(train_loader)
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {avg_loss:.4f} | "
        f"Accuracy: {accuracy:.2f}%"
    )


# ============================================================
# 20. TESTING
# ============================================================

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch[
            "input_ids"
        ].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        labels = batch[
            "labels"
        ].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )


# ============================================================
# 21. CALCULATE ACCURACY
# ============================================================

accuracy = (
    np.array(all_predictions)
    ==
    np.array(all_labels)
).mean() * 100


# ============================================================
# 22. FINAL RESULTS
# ============================================================

print("\n" + "=" * 60)
print("INDICBERT COMBINED BALANCED DATASET RESULTS")
print("=" * 60)

print(
    f"\nTest Accuracy: {accuracy:.2f}%"
)

print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Real",
            "Fake"
        ],
        digits=2
    )
)


# ============================================================
# 23. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("Confusion Matrix:")

print(cm)

print("\nMatrix:")
print("             Predicted")
print("             Real   Fake")

print(
    f"Actual Real   "
    f"{cm[0][0]:4d}   "
    f"{cm[0][1]:4d}"
)

print(
    f"Actual Fake   "
    f"{cm[1][0]:4d}   "
    f"{cm[1][1]:4d}"
)


# ============================================================
# 24. SAVE MODEL
# ============================================================

SAVE_PATH = (
    "/content/indicbert_combined_balanced"
)

model.save_pretrained(
    SAVE_PATH
)

tokenizer.save_pretrained(
    SAVE_PATH
)

print("\nModel saved to:")
print(SAVE_PATH)

Using device: cuda
GPU: Tesla T4

Dataset A shape: (14564, 2)
Dataset B shape: (5226, 2)

No duplicate removal performed.

COMBINED DATASET BEFORE BALANCING

Combined shape: (19790, 2)

Class distribution:
labels
0    13986
1     5804
Name: count, dtype: int64

Class 0 samples: 13986
Class 1 samples: 5804

Balancing both classes to: 5804

BALANCED COMBINED DATASET

Balanced shape: (11608, 2)

Balanced class distribution:
labels
0    5804
1    5804
Name: count, dtype: int64

TRAIN / TEST SPLIT

Training samples: 9286
Testing samples: 2322

Training distribution:
labels
0    4643
1    4643
Name: count, dtype: int64

Testing distribution:
labels
0    1161
1    1161
Name: count, dtype: int64


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

Map:   0%|          | 0/9286 [00:00<?, ? examples/s]

Map:   0%|          | 0/2322 [00:00<?, ? examples/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  135MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 1/3 | Loss: 0.2095 | Accuracy: 92.77%
Epoch 2/3 | Loss: 0.0981 | Accuracy: 96.67%
Epoch 3/3 | Loss: 0.0714 | Accuracy: 97.74%

INDICBERT COMBINED BALANCED DATASET RESULTS

Test Accuracy: 97.24%

Classification Report:
              precision    recall  f1-score   support

        Real       0.96      0.98      0.97      1161
        Fake       0.98      0.96      0.97      1161

    accuracy                           0.97      2322
   macro avg       0.97      0.97      0.97      2322
weighted avg       0.97      0.97      0.97      2322

Confusion Matrix:
[[1139   22]
 [  42 1119]]

Matrix:
             Predicted
             Real   Fake
Actual Real   1139     22
Actual Fake     42   1119


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to:
/content/indicbert_combined_balanced
